In [1]:
import sys
import os
sys.path.append('/root/capsule/code/beh_ephys_analysis')
from utils.beh_functions import parseSessionID, session_dirs, get_unit_tbl, get_session_tbl
from utils.plot_utils import shiftedColorMap, template_reorder, get_gradient_colors
from utils.opto_utils import opto_metrics, get_opto_tbl
from utils.ephys_functions import cross_corr_train, auto_corr_train, load_drift
import json
import matplotlib.pyplot as plt
import pandas as pd
import pickle
from aind_ephys_utils import align
from scipy.stats import wilcoxon
%matplotlib inline

In [2]:
def cal_opto_sigs(session, data_type):
    unit_tbl = get_unit_tbl(session, data_type)
    opto_tbl = get_opto_tbl(session, data_type, loc = 'soma')
    # loop through all conditions
    powers = opto_tbl['power'].unique().tolist()
    sites = opto_tbl['site'].unique().tolist()
    pre_posts = opto_tbl['pre_post'].unique().tolist()
    freqs = opto_tbl['freq'].unique().tolist()
    opto_sigs = pd.DataFrame()
    filter = unit_tbl['default_qc'] == 1
    unit_ids_focus = unit_tbl[filter]['unit_id'].unique().tolist()
    for unit in unit_ids_focus:
        # print(f"Processing unit {unit}...")
        spike_times = unit_tbl[unit_tbl['unit_id']== unit]['spike_times'].values[0]
        opto_tbl_curr = opto_tbl.copy()
        unit_drift = load_drift(session, unit)
        pulse_num = 5
        pre_win_ratio = 0.5
        post_win = 0.05
        if unit_drift is not None:
            if unit_drift['ephys_cut'][0] is not None:
                spike_times = spike_times[spike_times >= unit_drift['ephys_cut'][0]]
                opto_tbl_curr = opto_tbl_curr[opto_tbl_curr['time'] >= unit_drift['ephys_cut'][0]]
            if unit_drift['ephys_cut'][1] is not None:
                spike_times = spike_times[spike_times <= unit_drift['ephys_cut'][1]]
                opto_tbl_curr = opto_tbl_curr[opto_tbl_curr['time'] <= unit_drift['ephys_cut'][1]]
        for power_ind, power in enumerate(powers):
            for site_ind, site in enumerate(sites):
                for freq_ind, freq in enumerate(freqs):
                    for pre_post_ind, pre_post in enumerate(pre_posts):
                        # get the trials for this condition
                        trials = opto_tbl[(opto_tbl['power'] == power) & (opto_tbl['site'] == site) & (opto_tbl['pre_post'] == pre_post) & (opto_tbl['freq'] == freq)]
                        if len(trials) == 0:
                            # print(f"No trials for power {power}, site {site}, freq {freq}, pre_post {pre_post}")
                            continue
                        # get the pulse times
                        train_times = trials['time'].values
                        p_unit_condition = []
                        for pulse_ind in range(pulse_num):
                            pulse_times = train_times + (pulse_ind * 1/freq)  # assuming freq is in Hz and pulse_ind starts from 0
                            # get the opto signal
                            pre_win_counts = align.to_events(spike_times, pulse_times, [-1/freq * pre_win_ratio, 0], return_df=True)
                            post_win_counts = align.to_events(spike_times, pulse_times, [0, post_win], return_df=True)
                            pre_win_freq = [len(pre_win_counts[pre_win_counts['event_index'] == event_ind]) / (1/freq * pre_win_ratio) for event_ind in range(len(pulse_times))]
                            post_win_freq = [len(post_win_counts[post_win_counts['event_index'] == event_ind]) / (post_win) for event_ind in range(len(pulse_times))]
                            # paired non-parametric test
                            stat, p = wilcoxon(pre_win_freq, post_win_freq)
                            p_unit_condition.append(p)
                        # store the results
                        opto_sigs = pd.concat([opto_sigs, pd.DataFrame({
                                            'unit_id': [unit],
                                            'power': [power],
                                            'site': [site],
                                            'freq': [freq],
                                            'pre_post': [pre_post],
                                            'p_unit_condition': [p_unit_condition],  # wrap list of dicts in a list to keep in one cell
                                            'p_sig_count': [sum(p < 0.05 for p in p_unit_condition)],
                                        })], ignore_index=True)
    # save the results
    opto_sigs_file = os.path.join(session_dirs(session, data_type)[f'opto_dir_{data_type}'], f'{session}_opto_sigs.pkl')
    with open(opto_sigs_file, 'wb') as f:
        pickle.dump(opto_sigs, f)
    return opto_sigs

In [3]:
session_assets = pd.read_csv('/root/capsule/code/data_management/session_assets.csv')
session_list = [session for session in session_assets['session_id'] if isinstance(session, str)]

from joblib import Parallel, delayed
import os
import traceback

data_type = 'raw'

def process(session, data_type):
    print(f'Starting {session}')
    try:
        session_dir = session_dirs(session)

        opto_dir = session_dir.get(f'opto_dir_{data_type}', None)
        print(f'opto_dir_{data_type}:', opto_dir)

        if opto_dir is None:
            print(f'No opto_dir_{data_type} found for {session}')
            return

        out_file = os.path.join(opto_dir, f'{session}_opto_sigs.pkl')
        if os.path.exists(out_file):
            print(f'Already exists, skip: {out_file}')
            return

        cal_opto_sigs(session, data_type)
        plt.close('all')
        print(f'Finished {session}')

    except Exception as e:
        print(f'Error processing {session}: {type(e).__name__}: {e}')
        traceback.print_exc()
        plt.close('all')

Parallel(n_jobs=-8)(
    delayed(process)(session, data_type)
    for session in session_list
)

Starting behavior_792292_2025-08-15_14-15-47
Starting behavior_791174_2025-09-05_14-06-53
Starting behavior_814511_2025-10-16_13-44-32
opto_dir_raw: /root/capsule/scratch/792292/behavior_792292_2025-08-15_14-15-47/ephys/opto/raw
opto_dir_raw: /root/capsule/scratch/791174/behavior_791174_2025-09-05_14-06-53/ephys/opto/raw
opto_dir_raw: /root/capsule/scratch/814511/behavior_814511_2025-10-16_13-44-32/ephys/opto/raw
Starting behavior_792292_2025-08-13_14-47-17
Starting behavior_810888_2025-10-28_11-51-06
Starting behavior_814515_2025-10-24_13-22-07
opto_dir_raw: /root/capsule/scratch/792292/behavior_792292_2025-08-13_14-47-17/ephys/opto/raw
Starting behavior_810888_2025-10-31_12-39-56
opto_dir_raw: /root/capsule/scratch/810888/behavior_810888_2025-10-28_11-51-06/ephys/opto/raw
opto_dir_raw: /root/capsule/scratch/814515/behavior_814515_2025-10-24_13-22-07/ephys/opto/raw
opto_dir_raw: /root/capsule/scratch/810888/behavior_810888_2025-10-31_12-39-56/ephys/opto/raw
Error processing behavior_8

Traceback (most recent call last):
  File "/opt/conda/lib/python3.10/site-packages/pandas/core/indexes/base.py", line 3812, in get_loc
    return self._engine.get_loc(casted_key)
  File "pandas/_libs/index.pyx", line 167, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/index.pyx", line 196, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/hashtable_class_helper.pxi", line 7088, in pandas._libs.hashtable.PyObjectHashTable.get_item
  File "pandas/_libs/hashtable_class_helper.pxi", line 7096, in pandas._libs.hashtable.PyObjectHashTable.get_item
KeyError: 'unit_id'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/tmp/ipykernel_181266/2027967087.py", line 27, in process
  File "/tmp/ipykernel_181266/1871852358.py", line 11, in cal_opto_sigs
  File "/opt/conda/lib/python3.10/site-packages/pandas/core/frame.py", line 4113, in __getitem__
    indexer = self.columns.get_loc(key)
  File "/opt/conda/lib

Starting behavior_808655_2025-09-16_10-53-22
opto_dir_raw: /root/capsule/scratch/808655/behavior_808655_2025-09-16_10-53-22/ephys/opto/raw
Finished behavior_808655_2025-09-16_10-53-22
Starting behavior_808655_2025-09-18_13-56-51
opto_dir_raw: /root/capsule/scratch/808655/behavior_808655_2025-09-18_13-56-51/ephys/opto/raw
Finished behavior_792292_2025-08-15_14-15-47
Starting behavior_814515_2025-10-22_13-03-21
Starting behavior_814515_2025-10-23_14-41-04
opto_dir_raw: /root/capsule/scratch/814515/behavior_814515_2025-10-22_13-03-21/ephys/opto/raw
opto_dir_raw: /root/capsule/scratch/814515/behavior_814515_2025-10-23_14-41-04/ephys/opto/raw
Starting behavior_808650_2025-09-23_14-49-57


/opt/conda/lib/python3.10/site-packages/scipy/stats/_wilcoxon.py:172: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se
/opt/conda/lib/python3.10/site-packages/scipy/stats/_wilcoxon.py:172: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se


opto_dir_raw: /root/capsule/scratch/808650/behavior_808650_2025-09-23_14-49-57/ephys/opto/raw
Finished behavior_808655_2025-09-18_13-56-51
Starting behavior_808650_2025-09-24_12-38-00
opto_dir_raw: /root/capsule/scratch/808650/behavior_808650_2025-09-24_12-38-00/ephys/opto/raw
Finished behavior_814511_2025-10-16_13-44-32
Starting behavior_808650_2025-09-25_13-35-32
opto_dir_raw: /root/capsule/scratch/808650/behavior_808650_2025-09-25_13-35-32/ephys/opto/raw
Finished behavior_814515_2025-10-22_13-03-21
Starting behavior_808650_2025-09-26_11-51-57
opto_dir_raw: /root/capsule/scratch/808650/behavior_808650_2025-09-26_11-51-57/ephys/opto/raw
Finished behavior_810888_2025-10-28_11-51-06
Starting behavior_796387_2025-09-09_13-37-29
opto_dir_raw: /root/capsule/scratch/796387/behavior_796387_2025-09-09_13-37-29/ephys/opto/raw
Error processing behavior_796387_2025-09-09_13-37-29: KeyError: 'unit_id'
Starting behavior_792288_2025-08-26_11-41-18
opto_dir_raw: /root/capsule/scratch/792288/behavior

...

/opt/conda/lib/python3.10/site-packages/scipy/stats/_wilcoxon.py:172: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se


Finished behavior_792288_2025-08-26_11-41-18
Finished behavior_835451_2026-02-27_14-19-03
Finished behavior_835444_2026-02-19_13-08-36
Finished behavior_835444_2026-02-20_12-12-35


/opt/conda/lib/python3.10/site-packages/scipy/stats/_wilcoxon.py:172: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se


Finished behavior_808650_2025-09-24_12-38-00
Finished behavior_835444_2026-02-18_13-01-55
Finished behavior_835451_2026-02-25_13-19-37
Finished behavior_826159_2026-01-26_15-15-12


[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

In [68]:
class load_opto_sig():
    def __init__(self, session, data_type):
        self.session = session
        self.data_type = data_type
        self.opto_sigs = self.load_opto_sigs()

    def load_opto_sigs(self):
        opto_sigs_file = os.path.join(session_dirs(self.session)[f'opto_dir_{self.data_type}'], f'{self.session}_opto_sigs.pkl')
        if os.path.exists(opto_sigs_file):
            with open(opto_sigs_file, 'rb') as f:
                return pickle.load(f)
        else:
            print(f'No opto sigs found for {self.session}')
            return None

    def get_opto_sigs(self, unit):
        if self.opto_sigs is not None:
            unit_opto_sigs = self.opto_sigs[self.opto_sigs['unit_id'] == unit]
            if not unit_opto_sigs.empty:
                return unit_opto_sigs
            else:
                print(f'No opto sigs found for unit {unit} in session {self.session}')
                return None
        else:
            print(f'No opto sigs loaded for session {self.session}')
            return None

In [71]:
opto_sigs = load_opto_sig('behavior_754897_2025-03-13_11-20-42', 'curated')
curr_unit_sigs = opto_sigs.get_opto_sigs(10)
curr_unit_sigs

,unit_id,power,site,freq,pre_post,p_unit_condition,p_sig_count
7,10,50,surface_LC,5,pre,"[3.808961977046787e-05, 0.00014159510770021393...",5
8,10,50,surface_LC,5,post,"[0.008299215995280766, 0.0004087524092441004, ...",5
9,10,30,surface_LC,5,pre,"[0.0028375448887923154, 0.03582047350421736, 0...",5
10,10,30,surface_LC,5,post,"[0.0047315429964332155, 6.887215595255096e-05,...",5
11,10,10,surface_LC,5,post,"[0.806968367170738, 0.27257111883252894, 0.045...",3
12,10,20,surface_LC,5,post,"[0.05500883362926572, 0.006492857745083879, 0....",4
13,10,40,surface_LC,5,post,"[0.00010417979413217688, 0.0001438636130358712...",5


In [73]:
opto_response = opto_metrics('behavior_754897_2025-03-13_11-20-42', 'curated')
unit_opto_response = opto_response.load_unit(10)
unit_opto_response

,unit_id,resp_p,resp_p_bl,resp_lat,powers,sites,num_pulses,durations,freqs,stim_times,opto_pass,mean_p,euclidean_norm,correlation
56,10,0.40,0.305467,0.017459,20,surface_LC,5,4,5,post,True,0.175467,0.190139,0.982910
57,10,0.50,0.409505,0.015760,30,surface_LC,5,4,5,pre,True,0.302486,0.135288,0.986911
58,10,0.55,0.455467,0.016305,30,surface_LC,5,4,5,post,True,0.302486,0.151615,0.986995
59,10,0.50,0.405467,0.016863,30,surface_LC,5,4,5,post,True,0.302486,0.151615,0.986995
60,10,0.50,0.405467,0.013983,30,surface_LC,5,4,5,post,True,0.302486,0.151615,0.986995
61,10,0.45,0.355467,0.016462,30,surface_LC,5,4,5,post,True,0.302486,0.151615,0.986995
62,10,0.45,0.355467,0.013426,30,surface_LC,5,4,5,post,True,0.302486,0.151615,0.986995
63,10,0.70,0.605467,0.016810,40,surface_LC,5,4,5,post,True,0.455467,0.145236,0.986634
64,10,0.55,0.455467,0.017369,40,surface_LC,5,4,5,post,True,0.455467,0.145236,0.986634
65,10,0.45,0.355467,0.019567,40,surface_LC,5,4,5,post,True,0.455467,0.145236,0.986634
